# Notebook-First Exploration with kitchen

This notebook demonstrates the exploration workflow — no CLI commands needed to get started.

| Step | What | API |
|------|------|-----|
| 1 | Peek at data | `DataStore.preview()` |
| 2 | Try a quick idea inline | `kitchen.experiment()` |
| 3 | Run a full Trainer pipeline | `kitchen.init_run()` |
| 4 | Compare runs | `kitchen leaderboard` / `mlflow.search_runs()` |

**Prerequisites:** `pip install -e kitchen` (from the repo root, or `pip install rkoren-kitchen`)

The code below uses synthetic data so it runs out-of-the-box without any project setup.
In a real project, replace `project_dir` with your actual project root and skip the
data-generation cells.

## Setup

In [ ]:
import os
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import kitchen
from kitchen.store import DataStore

# -------------------------------------------------------------------------
# Temporary project root — replace with your real project directory in
# a live project (e.g. project_dir = Path("."))
# -------------------------------------------------------------------------
project_dir = Path(tempfile.mkdtemp())
store = DataStore(root=project_dir)
store.raw_dir.mkdir(parents=True)
store.processed_dir.mkdir(parents=True)

# Route all MLflow runs to a SQLite file inside the temp dir so this
# notebook doesn't pollute your project's mlruns.db.
# In a real project, this is set via .env or params.yaml and you can
# remove this line entirely.
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{project_dir}/mlruns.db"

# -------------------------------------------------------------------------
# Synthetic dataset: 500 rows, 4 numeric features, binary target
# Replace this block with store.preview("data.csv") in a real project.
# -------------------------------------------------------------------------
rng = np.random.default_rng(42)
n = 500
raw = pd.DataFrame({
    "feature_a": rng.normal(0.0, 1.0, n),
    "feature_b": rng.normal(0.0, 1.0, n),
    "feature_c": rng.uniform(0.0, 10.0, n),
    "feature_d": rng.choice([0, 1, 2], n).astype(float),
    "target":    (rng.normal(0.0, 1.0, n) > 0.0).astype(int),
})
raw.to_csv(store.raw_dir / "data.csv", index=False)
raw.to_parquet(store.processed_dir / "features.parquet", index=False)

print(f"Project root: {project_dir}")
print(f"Raw files:       {store.list('raw')}")
print(f"Processed files: {store.list('processed')}")

## Step 1 — Load data with `DataStore.preview()`

`preview(filename, n=5)` searches `processed/` first, then `raw/`, and returns the
first `n` rows.  No path juggling — just the filename.

In [ ]:
# Peek at raw data
store.preview("data.csv")

In [ ]:
# Peek at processed features — preview() finds it in processed/ automatically
store.preview("features.parquet")

## Step 2 — Try a quick idea with `kitchen.experiment()`

`kitchen.experiment()` is the zero-ceremony path:
- Write model code directly in a cell — no `Trainer` subclass required
- Auto-discovers `params.yaml` from the working directory (falls back to `sqlite:///mlruns.db`)
- Runs appear in `kitchen leaderboard` and the MLflow UI immediately

Ideal for early iteration when you don't yet have a reusable `Trainer`.

In [ ]:
df = store.load_parquet("features.parquet")
X = df.drop(columns=["target"])
y = df["target"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Idea A: logistic regression with C=1.0
with kitchen.experiment(
    "exploration-demo",
    run_name="logistic-C1",
    params={"model": "logistic", "C": 1.0},
) as run_a:
    model_a = LogisticRegression(C=1.0, max_iter=200)
    model_a.fit(X_train, y_train)
    acc_a = accuracy_score(y_val, model_a.predict(X_val))
    run_a.log(val_accuracy=acc_a)

print(f"run_id: {run_a.run_id}")
print(f"val_accuracy: {acc_a:.4f}")

In [ ]:
# Idea B: tighter regularisation — one line change, separate run
with kitchen.experiment(
    "exploration-demo",
    run_name="logistic-C0.1",
    params={"model": "logistic", "C": 0.1},
) as run_b:
    model_b = LogisticRegression(C=0.1, max_iter=200)
    model_b.fit(X_train, y_train)
    acc_b = accuracy_score(y_val, model_b.predict(X_val))
    run_b.log(val_accuracy=acc_b)

print(f"run_id: {run_b.run_id}")
print(f"val_accuracy: {acc_b:.4f}")

## Step 3 — Structured run with `kitchen.init_run()`

Once you have a project `Trainer` class, use `kitchen.init_run()` to inject MLflow
tracking automatically — the same run that `kitchen run train` would open.

This lets you iterate on your `Trainer` in the notebook and see the result in the
same leaderboard as your CLI runs, with zero code duplication.

In [ ]:
import mlflow

from kitchen.steps import Trainer


class SimpleTrainer(Trainer):
    """Minimal Trainer: logistic regression with configurable regularisation.

    In a real project this class lives in src/train/run.py and is the same
    code that `kitchen run train` executes.
    """

    def fit(self, df: pd.DataFrame, params: dict) -> object:
        X = df.drop(columns=["target"])
        y = df["target"]
        X_tr, X_vl, y_tr, y_vl = train_test_split(X, y, test_size=0.2, random_state=42)

        c_val = float(params.get("model", {}).get("C", 1.0))
        model = LogisticRegression(C=c_val, max_iter=300)
        model.fit(X_tr, y_tr)

        val_accuracy = accuracy_score(y_vl, model.predict(X_vl))
        mlflow.log_metric("val_accuracy", val_accuracy)
        print(f"  val_accuracy: {val_accuracy:.4f}")
        return model

In [ ]:
# Params mirror what params.yaml would contain in a real project
run_params = {
    "experiment": "exploration-demo",
    "model": {"C": 0.5},
    "processed_file": "features.parquet",
}

with kitchen.init_run(run_params, run_name="structured-C0.5") as tracker:
    SimpleTrainer().run(store, tracker, run_params)

print("Run logged — visible in kitchen leaderboard")

## Step 4 — Compare runs

All three runs are now tracked.  From a **terminal** (in your project directory):

```bash
# Ranked leaderboard
kitchen leaderboard

# Add param columns
kitchen leaderboard --show-params C

# Diff the best and worst run
kitchen diff <run_id_a> <run_id_b>

# Open the MLflow UI
kitchen ui
```

**Metric naming matters.** `kitchen leaderboard` ranks by the metric in your `thresholds:` section (e.g. `loto_brier`). Log under that exact name — a run that logs `val_brier` when the threshold is `loto_brier` won't show up in the default leaderboard. `kitchen leaderboard --metric val_brier` still finds it, and the command points out the mismatch when no run matches the configured metric.

Or query results directly from this notebook:

In [ ]:
import mlflow

runs = mlflow.search_runs(
    experiment_names=["exploration-demo"],
    order_by=["metrics.val_accuracy DESC"],
)

# Show the columns that are always present; param column names vary by
# how you logged them (flat params.C vs nested params.model.C).
display_cols = ["run_id", "tags.mlflow.runName", "metrics.val_accuracy"]
display_cols += [c for c in runs.columns if c.startswith("params.")]
runs[display_cols].head()

### Promote the best run

Once you've found a run worth keeping, promote it as champion from the terminal:

```bash
# Promote the metric leader automatically
kitchen promote val_accuracy

# Or target a specific run by ID
kitchen promote --run-id <run_id>
```

The champion is then loadable anywhere with:

```python
import mlflow.sklearn
model = mlflow.sklearn.load_model("models:/my-project@champion")
```